# 01 - Business Understanding & Pengumpulan Data

## Tujuan
Mengambil data skripsi mahasiswa Program Studi Sistem Informasi UIN Raden Fatah
Palembang periode 2021-2025 dari repositori resmi institusi.

## Asal (Input)
`data/raw/html_source/2021.html` ... `2025.html` - halaman indeks repositori
EPrints yang disimpan dengan batas waktu **19 September 2026**.

## Keluaran (Output)
`data/raw/thesis_scraped.csv` - 312 dokumen unik
(id, year, author, title, abstract, url)

## Posisi dalam CRISP-DM
Fase pertama CRISP-DM adalah **Business Understanding**: memahami tujuan dan
ruang lingkup riset. Notebook ini mendokumentasikan kebutuhan Kaprodi
(pemetaan topik otomatis) dan mahasiswa (menemukan novelty) yang menjadi
dasar seluruh analisis.

In [1]:
import sys, io as _io
from pathlib import Path
_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(_ROOT))
try:
    sys.stdout.reconfigure(encoding='utf-8', errors='replace')
except Exception:
    pass

import pandas as pd, numpy as np
from src import config as C
C.ensure_dirs()
print(C.describe())

KONFIGURASI PROYEK
  Repo          : C:\Users\HAJIWAHYU\Downloads\lda-topik-skripsi-si-uinrf-main
  Dokumen akhir : 285 (2021, 2022, 2023, 2024, 2025)
  Vocabulary    : 1849
  Model LDA     : K=8, alpha=asymmetric, eta=auto, passes=30, iterations=500, seed=42
  Inferensi     : Online Variational Bayes (bukan Gibbs Sampling)
  C_V resmi     : 0.4256
  Cutoff data   : 19 September 2026

## Rumusan Masalah yang Menjadi Acuan

1. Bagaimana memetakan topik skripsi dengan LDA?
2. Bagaimana memprediksi tren distribusi topik dengan WMA?
3. Bagaimana membangun dashboard interaktif sebagai alat bantu keputusan?

## Langkah 1 - Jalankan Pengumpulan Data

In [2]:
from src import scraping

n = scraping.run(verbose=True)
print()
print(f'Total dokumen : {n}')
print(f'Nilai harapan : {C.TOTAL_DOCS_RAW}')

  [2021]  31 skripsi ditemukan
  [2022]  90 skripsi ditemukan
  [2023]  48 skripsi ditemukan
  [2024]  51 skripsi ditemukan
  [2025]  92 skripsi ditemukan

  Total sebelum deduplikasi: 312
  Duplikat dibuang : 0
  Dokumen unik    : 312

[OK] 312 skripsi -> data\raw\thesis_scraped.csv

Total dokumen : 312
Nilai harapan : 312

## Langkah 2 - Distribusi Dokumen per Tahun

In [3]:
raw_df = pd.read_csv(C.SCRAPED_CSV)
print(raw_df.groupby('year').size().rename('jumlah').to_string())
print()
print(f'Total: {len(raw_df)} dokumen')

year
2021    31
2022    90
2023    48
2024    51
2025    92

Total: 312 dokumen

## Langkah 3 - Cek Kelengkapan Field

In [4]:
for col in ['id', 'year', 'author', 'title', 'abstract']:
    filled = raw_df[col].notna().sum()
    print(f'  {col:10} terisi {filled:3d}/{len(raw_df)}  ({filled/len(raw_df)*100:5.1f}%)')

  id         terisi 312/312  (100.0%)
  year       terisi 312/312  (100.0%)
  author     terisi   0/312  (  0.0%)
  title      terisi 312/312  (100.0%)
  abstract   terisi   0/312  (  0.0%)

## Ringkasan Fase 01 - Business Understanding

| Hasil | Nilai |
|:---|:---|
| Dokumen terambil | **312 skripsi** |
| Rentang tahun | 2021-2025 |
| Cutoff | 19 September 2026 |
| Sumber | Repositori EPrints UIN Raden Fatah |

Kebutuhan Kaprodi dan mahasiswa terjawab melalui pengumpulan data ini.
Langkah berikutnya adalah memahami struktur dan kualitas data.

## Peta Artefak

| Artefak | Dibuat oleh | Consumed by |
|:---|:---|:---|
| `data/raw/html_source/2021-2025.html` | Save Page As repositori | Notebook 01 |
| `data/raw/thesis_scraped.csv` | Notebook 01 | Notebook 02, 03 |